In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install rdkit==2024.9.5
!pip install torch_geometric==2.5.3

In [2]:
import os
import sys
import torch
from torch.utils.data import DataLoader
doc_name = "/content/drive/MyDrive/HeckLit-Code-Colab"
sys.path.append(doc_name)
from utils.rxn import *
from utils.molecule import *
from utils.dataset_analysis import *
from models.DeepLearnModel import *
import time
from tqdm import tqdm
import datetime
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# SSTS method based on data statistics
# Dimension Reduction
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
data = pd.read_excel("%s/data/Heck/Heck_fp.xlsx" % doc_name)
X = list()
for i in range(data.shape[0]):
  drfp = read_drfp(data.loc[i]["drfp"])
  X.append(drfp)
# Standard
X = np.array(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
# plot to determine the number of subsets
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.7)
plt.title('PCA Result')
plt.xlabel('PCA1')
plt.ylabel('PCA2')
plt.grid(True, alpha=0.7, linestyle="--")
plt.show()

In [ ]:
# Cluster Method to determine the number of subsets
from sklearn.cluster import KMeans
n_clusters = int(input("By PCA plot, the number of the subsets should be:"))
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
kmeans.fit(X_pca)

# Get sample label
labels = kmeans.labels_

# plot again
plt.figure(figsize=(8, 6))
for i in range(n_clusters):
    plt.scatter(X_pca[labels == i, 0], X_pca[labels == i, 1], label=f'Cluster {i}', alpha=0.7)

centers = kmeans.cluster_centers_
labels = pd.DataFrame(labels, columns=["label"])
plt.scatter(centers[:, 0], centers[:, 1], s=200, c='black', marker='x', label='Centers')
plt.title('PCA + KMeans')
plt.xlabel('PCA1')
plt.ylabel('PCA2')
plt.legend()
plt.grid(True, alpha=0.7, linestyle="--")
plt.savefig("%s/figures/SubsetSplitn_Cluster_%s.png" % (doc_name, n_clusters))
plt.show()

# testing
rs_list = [1,2,3,4,5]
columns = ['test_R2','test_RMSE','test_MAE']
eval_metrics = np.zeros((1+len(rs_list), len(columns)))
index = []
for rs in rs_list:
  index.append("%s" % rs)
index.append("avg±std")
eval_metrics = pd.DataFrame(eval_metrics, columns=columns, index=index)

for rs in rs_list:
    # 1. import data
    data = pd.read_excel("%s/data/Heck/Heck_fp.xlsx" % doc_name)
    # Combine with data category
    data = pd.concat([data, labels], axis=1)
    random_state = rs
    data = data.sample(random_state=random_state, frac=1).reset_index(drop=True)
    subset_list = list()
    len_drfp = 0
    for i in range(n_clusters):
        subset_list.append([])
    for i in tqdm(range(data.shape[0])):
        drfp = torch.tensor(read_drfp(data.loc[i]["drfp"]), dtype=torch.float32)
        len_drfp = drfp.shape[0]
        y = data.loc[i]["Yield"] / 100
        label = int(data.loc[i]["label"])
        subset_list[label].append([drfp, y])

    # report
    dir_path = "%s/exp/Heck_Other/Heck_SSTS_nCluster=%s_rs=%s_%s" % (doc_name, n_clusters, rs, datetime.datetime.now())
    os.mkdir("%s" % dir_path)
    f = open("%s/Model_Training_Report.txt" % dir_path, mode="w")

    # predicted value list
    pred_list = []
    true_list = []

    for i in range(len(subset_list)):
        # dataset split
        subset = subset_list[i]
        ratio = 0.8
        batch_size_ratio = 0.1
        batch_size = int(len(subset) * batch_size_ratio)
        batch = len(subset)
        train_set = subset[0: int(ratio * batch)]
        test_set = subset[int(ratio * batch) + 1:]

        # data_loader
        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, drop_last=True)
        test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=True, drop_last=True)

        train_R2 = list()
        train_RMSE = list()
        train_MAE = list()
        test_R2 = list()
        test_RMSE = list()
        test_MAE = list()
        pred = list()
        true = list()

        # 3. training of the model
        # params
        t = 2000
        lr = 1e-4

        # model
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = ANN(input_size=len_drfp).to(device)
        opti = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
        criterion = nn.MSELoss()

        # writedown params
        f.write("Data Class=%s\n" % i)
        f.write("params:\n")
        f.write("Data Class=%s\n" % i)
        f.write("random_state=%s\n" % random_state)
        f.write("ratio=%s\n" % ratio)
        f.write("batch_size=%s\n" % batch_size)
        f.write("t=%s\n" % t)
        f.write("lr=%s\n" % lr)

        # Training
        # best performance
        best = [0, 0, 0, 0, 0, 0, [], []]  # train_R2, train_RMSE, train_MAE, test_R2, test_RMSE, test_MAE, test_predict, test_true

        f.write("\nStart training\n")

        for epoch in tqdm(range(t)):
            # Training
            global_loss = torch.tensor([0.])

            for data in train_loader:
                x = data[:-1][0].to(device)
                y = torch.unsqueeze(data[-1], dim=1).to(device)
                loss = criterion(model.forward(x).float(), y.float())
                opti.zero_grad()
                loss.backward()
                opti.step()
                global_loss += loss.item()

            # record of loss during training
            # performance in train set
            with torch.no_grad():
                pred = list()
                true = list()
                for data in train_loader:
                    x = data[:-1][0].to(device)
                    tr = torch.unsqueeze(data[-1], dim=1).to(device)
                    pr = list(model.forward(x).cpu().detach().numpy())
                    pred += pr
                    true += list(tr.cpu().detach().numpy())
                train_R2.append(R2(np.array(pred), np.array(true)))
                train_RMSE.append(RMSE(np.array(pred), np.array(true)))
                train_MAE.append(MAE(np.array(pred), np.array(true)))

            # performance in test set
            with torch.no_grad():
                pred = list()
                true = list()
                for data in test_loader:
                    x = data[:-1][0].to(device)
                    tr = torch.unsqueeze(data[-1], dim=1).to(device)
                    pr = list(model.forward(x).cpu().detach().numpy())
                    pred += pr
                    true += list(tr.cpu().detach().numpy())
                test_R2.append(R2(np.array(pred), np.array(true)))
                test_RMSE.append(RMSE(np.array(pred), np.array(true)))
                test_MAE.append(MAE(np.array(pred), np.array(true)))

                if epoch == 0 or test_R2[-1] >= best[3]:
                    best = [train_R2[-1], train_RMSE[-1], train_MAE[-1], test_R2[-1], test_RMSE[-1], test_MAE[-1], pred, true]

            # write report
            f.write("Epoch:%d loss: %f, R2:train set %.3f\ttest set %.3f\n" % (
            epoch + 1, global_loss / batch_size, train_R2[-1], test_R2[-1]))

        pred_list.append(best[-2])
        true_list.append(best[-1])

        # 4.Evaluation
        f.write("\n")
        # Performance in train set
        f.write("[rs=%s Subset:%s] R2 of train set is:%.3f+-%f\tbest:%f\n" % (rs, i,
        np.array(train_R2[-10:]).mean(), np.array(train_R2[-10:]).std(), best[0]))
        f.write("[rs=%s Subset:%s] RMSE of train set is: %.3f+-%f\tbest:%f\n" % (rs, i,
        np.array(train_RMSE[-10:]).mean(), np.array(train_RMSE[-10:]).std(), best[1]))
        f.write("[rs=%s Subset:%s] MAE of train set is: %.3f+-%f\tbest:%f\n" % (rs, i,
        np.array(train_MAE[-10:]).mean(), np.array(train_MAE[-10:]).std(), best[2]))

        # Performance in test set
        f.write("[rs=%s Subset:%s] R2 of test set is:%.3f+-%.3f\tbest:%f\n" % (rs, i,
        np.array(test_R2[-10:]).mean(), np.array(test_R2[-10:]).std(), best[3]))
        f.write("[rs=%s Subset:%s] RMSE of test set is: %.3f+-%f\tbest:%f\n" % (rs, i,
        np.array(test_RMSE[-10:]).mean(), np.array(test_RMSE[-10:]).std(), best[4]))
        f.write("[rs=%s Subset:%s] MAE of test set is: %.3f+-%f\tbest:%f\n" % (rs, i,
        np.array(test_MAE[-10:]).mean(), np.array(test_MAE[-10:]).std(), best[5]))

        # Performance in train set
        print("[rs=%s Subset:%s] R2 of train set is:%.3f+-%f\tbest:%f\n" % (rs, i,
        np.array(train_R2[-10:]).mean(), np.array(train_R2[-10:]).std(), best[0]))
        print("[rs=%s Subset:%s] RMSE of train set is: %.3f+-%f\tbest:%f\n" % (rs, i,
        np.array(train_RMSE[-10:]).mean(), np.array(train_RMSE[-10:]).std(), best[1]))

        # Performance in test set
        print("[rs=%s Subset:%s] R2 of test set is:%.3f+-%.3f\tbest:%f\n" % (rs, i,
        np.array(test_R2[-10:]).mean(), np.array(test_R2[-10:]).std(), best[3]))
        print("[rs=%s Subset:%s] RMSE of test set is: %.3f+-%f\tbest:%f\n" % (rs, i,
        np.array(test_RMSE[-10:]).mean(), np.array(test_RMSE[-10:]).std(), best[4]))


        # 5.Figure
        import matplotlib.pyplot as plt
        import seaborn as sns

        fig = plt.figure(dpi=300, figsize=(10, 7))

        # Training Fig
        plt.subplot(1, 1, 1)
        steps = np.linspace(1, t, t)
        plt.plot(steps, train_R2, color=[236 / 255, 164 / 255, 124 / 255])
        plt.plot(steps, test_R2, color=[117 / 255, 157 / 255, 219 / 255])
        # Beautify
        plt.legend(["train set R$^2$", "test set R$^2$"], loc="upper left", prop={'size': 8})
        plt.xlabel("Epoch", fontsize=10)
        plt.ylabel("R$^2$ value", fontsize=10)
        plt.title("The R$^2$ value during training", fontsize=13)

        fig.suptitle("SSTS for HeckLit subset(label=%s)" % i, fontsize=16)
        plt.tight_layout()
        plt.savefig("%s/Performance(label=%s).png" % (dir_path, i))
        plt.show()


    # Calculate SSTS method evaluation metrics
    pred = []
    for i in pred_list:
      for j in i:
        pred.append(j)
    pred = np.array(pred)
    true = []
    for i in true_list:
      for j in i:
        true.append(j)
    true = np.array(true)
    true = np.array(true)
    r2 = R2(pred, true)
    rmse = RMSE(pred, true)
    mae = MAE(pred, true)
    eval_metrics.loc["%s" % rs]["test_R2"] = r2
    eval_metrics.loc["%s" % rs]["test_RMSE"] = rmse
    eval_metrics.loc["%s" % rs]["test_MAE"] = mae
    print("rs=%s\tR2=%.3f\tRMSE=%.4f\tMAE=%.4f\n" % (rs, r2, rmse, mae))
    f.write("\nSummary:\n")
    f.write("R2=%.3f\nRMSE=%.4f\nMAE=%.4f\n" % (r2, rmse, mae))
    f.close()

# Evaluation metrics report
for i in range(len(rs_list), eval_metrics.shape[0], len(rs_list)+1):
  for j in range(eval_metrics.shape[1]):
    eval_metrics.iloc[i,j] = "%.4f ± %.4f" % (eval_metrics.iloc[i-len(rs_list):i-1,j].mean(), eval_metrics.iloc[i-len(rs_list):i-1,j].std())
eval_metrics.to_csv("%s/exp/Heck_Other/SSTS_report_nCluster_%s_%s.csv" % (doc_name, n_clusters, datetime.datetime.now()))
print(eval_metrics)

In [ ]:
# SSTS method based on chemical knowledge
# Subset Split basis
inter_intra = True
alpha_beta = True
assert not (not inter_intra and not alpha_beta) # alpha_beta and inter_intra can not be False at the same time

# PCA Visualization of different chemical split method
data = pd.read_excel("%s/data/Heck/Heck_fp.xlsx" % doc_name)
# Get the label of rxn based on reaction property
labels = []
rxn_list = df_to_rxn_list(data)

label_type = 1
if inter_intra:
  label_type *= 2
if alpha_beta:
  label_type *= 3
label_type = [i for i in range(label_type)]

for i in tqdm(range(len(rxn_list))):
  rxn = rxn_list[i]
  # Intramolecular
  if len(rxn.reactants) == 1:
    if alpha_beta:
      Type_Add = HeckIntra_Classify(rxn)[0]
      if Type_Add == "Alpha":
        labels.append(0)
      if Type_Add == "Beta":
        labels.append(1)
      if Type_Add == "*" or Type_Add is None:
        labels.append(2)
    else:
      labels.append(0)

  # Intermolecular
  if len(rxn.reactants) == 2:
    if alpha_beta:
      Type_Add = HeckInter_Classify(rxn)[0]
      if inter_intra:
        if Type_Add == "Alpha":
          labels.append(3)
        if Type_Add == "Beta":
          labels.append(4)
        if Type_Add == "*" or Type_Add is None:
          labels.append(5)
      else:
        if Type_Add == "Alpha":
          labels.append(0)
        if Type_Add == "Beta":
          labels.append(1)
        if Type_Add == "*":
          labels.append(2)
    else:
      labels.append(1)

labels = pd.DataFrame(np.array(labels).reshape(-1, 1), columns=["label"])
assert data.shape[0] == labels.shape[0]

# PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
X = list()
for i in range(data.shape[0]):
  drfp = read_drfp(data.loc[i]["drfp"])
  X.append(drfp)
X = np.array(X)
# Standard
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Plot the figure
plt.figure(figsize=(8, 6), dpi=300)
for i in range(len(label_type)):
    label_index = labels.index[labels['label'] == i].tolist()
    if label_index:
        plt.scatter(X_pca[label_index, 0], X_pca[label_index, 1], label=f'Cluster {i}', alpha=0.5)
plt.title('PCA Visualization')
plt.xlabel('PCA1')
plt.ylabel('PCA2')
plt.legend()
plt.grid(True, alpha=0.7, linestyle="--")
plt.savefig("%s/figures/SSTS_PCA_Chem(I=%s,R=%s).png" % (doc_name, inter_intra, alpha_beta))
plt.show()

# Testing
rs_list = [1,2,3,4,5]
columns = ['test_R2','test_RMSE','test_MAE']
eval_metrics = np.zeros((1+len(rs_list), len(columns)))
index = []
for rs in rs_list:
  index.append("%s" % rs)
index.append("avg±std")
eval_metrics = pd.DataFrame(eval_metrics, columns=columns, index=index)

for rs in rs_list:
    # 1. import data
    data = pd.read_excel("%s/data/Heck/Heck_fp.xlsx" % doc_name)
    # Get the label of rxn based on reaction property
    labels = []
    rxn_list = df_to_rxn_list(data)

    label_type = 1
    if inter_intra:
      label_type *= 2
    if alpha_beta:
      label_type *= 3
    label_type = [i for i in range(label_type)]

    for i in tqdm(range(len(rxn_list))):
      rxn = rxn_list[i]
      # Intramolecular
      if len(rxn.reactants) == 1:
        if alpha_beta:
          Type_Add = HeckIntra_Classify(rxn)[0]
          if Type_Add == "Alpha":
            labels.append(0)
          if Type_Add == "Beta":
            labels.append(1)
          if Type_Add == "*" or Type_Add is None:
            labels.append(2)
        else:
          labels.append(0)

      # Intermolecular
      if len(rxn.reactants) == 2:
        if alpha_beta:
          Type_Add = HeckInter_Classify(rxn)[0]
          if inter_intra:
            if Type_Add == "Alpha":
              labels.append(3)
            if Type_Add == "Beta":
              labels.append(4)
            if Type_Add == "*" or Type_Add is None:
              labels.append(5)
          else:
            if Type_Add == "Alpha":
              labels.append(0)
            if Type_Add == "Beta":
              labels.append(1)
            if Type_Add == "*":
              labels.append(2)
        else:
          labels.append(1)

    labels = pd.DataFrame(np.array(labels).reshape(-1, 1), columns=["label"])
    assert data.shape[0] == labels.shape[0]

    # Combine with data category
    data = pd.concat([data, labels], axis=1)
    random_state = rs
    data = data.sample(random_state=random_state, frac=1).reset_index(drop=True)
    subset_list = []
    for i in range(len(label_type)):
      subset_list.append([])
    len_drfp = 0
    for i in tqdm(range(data.shape[0])):
        drfp = torch.tensor(read_drfp(data.loc[i]["drfp"]), dtype=torch.float32)
        len_drfp = drfp.shape[0]
        y = data.loc[i]["Yield"] / 100
        label = int(data.loc[i]["label"])
        subset_list[label].append([drfp, y])

    # report
    dir_path = "%s/exp/Heck_Other/Heck_SSTS_Chem(I=%s,R=%s)_rs=%s_%s" % (doc_name, inter_intra, alpha_beta, rs, datetime.datetime.now())
    os.mkdir("%s" % dir_path)
    f = open("%s/Model_Training_Report.txt" % dir_path, mode="w")

    # predicted value list
    pred_list = []
    true_list = []

    for i in range(len(subset_list)):
        # dataset split
        subset = subset_list[i]
        ratio = 0.8
        batch_size_ratio = 0.1
        batch_size = int(len(subset) * batch_size_ratio)
        batch = len(subset)
        train_set = subset[0: int(ratio * batch)]
        test_set = subset[int(ratio * batch) + 1:]

        # data_loader
        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, drop_last=True)
        test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=True, drop_last=True)

        train_R2 = list()
        train_RMSE = list()
        train_MAE = list()
        test_R2 = list()
        test_RMSE = list()
        test_MAE = list()
        pred = list()
        true = list()

        # 3. training of the model
        # params
        t = 2000
        lr = 1e-4

        # model
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = ANN(input_size=len_drfp).to(device)
        opti = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
        criterion = nn.MSELoss()

        # writedown params
        f.write("Data Class=%s\n" % i)
        f.write("params:\n")
        f.write("Data Class=%s\n" % i)
        f.write("random_state=%s\n" % random_state)
        f.write("ratio=%s\n" % ratio)
        f.write("batch_size=%s\n" % batch_size)
        f.write("t=%s\n" % t)
        f.write("lr=%s\n" % lr)

        # Training
        # best performance
        best = [0, 0, 0, 0, 0, 0, [], []]  # train_R2, train_RMSE, train_MAE, test_R2, test_RMSE, test_MAE, test_predict, test_true

        f.write("\nStart training\n")

        for epoch in tqdm(range(t)):
            # Training
            global_loss = torch.tensor([0.])

            for data in train_loader:
                x = data[:-1][0].to(device)
                y = torch.unsqueeze(data[-1], dim=1).to(device)
                loss = criterion(model.forward(x).float(), y.float())
                opti.zero_grad()
                loss.backward()
                opti.step()
                global_loss += loss.item()

            # record of loss during training
            # performance in train set
            with torch.no_grad():
                pred = list()
                true = list()
                for data in train_loader:
                    x = data[:-1][0].to(device)
                    tr = torch.unsqueeze(data[-1], dim=1).to(device)
                    pr = list(model.forward(x).cpu().detach().numpy())
                    pred += pr
                    true += list(tr.cpu().detach().numpy())
                train_R2.append(R2(np.array(pred), np.array(true)))
                train_RMSE.append(RMSE(np.array(pred), np.array(true)))
                train_MAE.append(MAE(np.array(pred), np.array(true)))

            # performance in test set
            with torch.no_grad():
                pred = list()
                true = list()
                for data in test_loader:
                    x = data[:-1][0].to(device)
                    tr = torch.unsqueeze(data[-1], dim=1).to(device)
                    pr = list(model.forward(x).cpu().detach().numpy())
                    pred += pr
                    true += list(tr.cpu().detach().numpy())
                test_R2.append(R2(np.array(pred), np.array(true)))
                test_RMSE.append(RMSE(np.array(pred), np.array(true)))
                test_MAE.append(MAE(np.array(pred), np.array(true)))

                if epoch == 0 or test_R2[-1] >= best[3]:
                    best = [train_R2[-1], train_RMSE[-1], train_MAE[-1], test_R2[-1], test_RMSE[-1], test_MAE[-1], pred, true]

            # write report
            f.write("Epoch:%d loss: %f, R2:train set %.3f\ttest set %.3f\n" % (
            epoch + 1, global_loss / batch_size, train_R2[-1], test_R2[-1]))

        pred_list.append(best[-2])
        true_list.append(best[-1])

        # 4.Evaluation
        f.write("\n")
        # Performance in train set
        f.write("[rs=%s Subset:%s] R2 of train set is:%.3f+-%f\tbest:%f\n" % (rs, i,
        np.array(train_R2[-10:]).mean(), np.array(train_R2[-10:]).std(), best[0]))
        f.write("[rs=%s Subset:%s] RMSE of train set is: %.3f+-%f\tbest:%f\n" % (rs, i,
        np.array(train_RMSE[-10:]).mean(), np.array(train_RMSE[-10:]).std(), best[1]))
        f.write("[rs=%s Subset:%s] MAE of train set is: %.3f+-%f\tbest:%f\n" % (rs, i,
        np.array(train_MAE[-10:]).mean(), np.array(train_MAE[-10:]).std(), best[2]))

        # Performance in test set
        f.write("[rs=%s Subset:%s] R2 of test set is:%.3f+-%.3f\tbest:%f\n" % (rs, i,
        np.array(test_R2[-10:]).mean(), np.array(test_R2[-10:]).std(), best[3]))
        f.write("[rs=%s Subset:%s] RMSE of test set is: %.3f+-%f\tbest:%f\n" % (rs, i,
        np.array(test_RMSE[-10:]).mean(), np.array(test_RMSE[-10:]).std(), best[4]))
        f.write("[rs=%s Subset:%s] MAE of test set is: %.3f+-%f\tbest:%f\n" % (rs, i,
        np.array(test_MAE[-10:]).mean(), np.array(test_MAE[-10:]).std(), best[5]))

        # Performance in train set
        print("[rs=%s Subset:%s] R2 of train set is:%.3f+-%f\tbest:%f\n" % (rs, i,
        np.array(train_R2[-10:]).mean(), np.array(train_R2[-10:]).std(), best[0]))
        print("[rs=%s Subset:%s] RMSE of train set is: %.3f+-%f\tbest:%f\n" % (rs, i,
        np.array(train_RMSE[-10:]).mean(), np.array(train_RMSE[-10:]).std(), best[1]))

        # Performance in test set
        print("[rs=%s Subset:%s] R2 of test set is:%.3f+-%.3f\tbest:%f\n" % (rs, i,
        np.array(test_R2[-10:]).mean(), np.array(test_R2[-10:]).std(), best[3]))
        print("[rs=%s Subset:%s] RMSE of test set is: %.3f+-%f\tbest:%f\n" % (rs, i,
        np.array(test_RMSE[-10:]).mean(), np.array(test_RMSE[-10:]).std(), best[4]))


        # 5.Figure
        import matplotlib.pyplot as plt
        import seaborn as sns

        fig = plt.figure(dpi=300, figsize=(10, 7))

        # Training Fig
        plt.subplot(1, 1, 1)
        steps = np.linspace(1, t, t)
        plt.plot(steps, train_R2, color=[236 / 255, 164 / 255, 124 / 255])
        plt.plot(steps, test_R2, color=[117 / 255, 157 / 255, 219 / 255])
        # Beautify
        plt.legend(["train set R$^2$", "test set R$^2$"], loc="upper left", prop={'size': 8})
        plt.xlabel("Epoch", fontsize=10)
        plt.ylabel("R$^2$ value", fontsize=10)
        plt.title("The R$^2$ value during training", fontsize=13)

        fig.suptitle("SSTS for HeckLit subset(label=%s)" % i, fontsize=16)
        plt.tight_layout()
        plt.savefig("%s/Performance(label=%s).png" % (dir_path, i))
        plt.show()


    # Calculate SSTS method evaluation metrics
    pred = []
    for i in pred_list:
      for j in i:
        pred.append(j)
    pred = np.array(pred)
    true = []
    for i in true_list:
      for j in i:
        true.append(j)
    true = np.array(true)
    true = np.array(true)
    r2 = R2(pred, true)
    rmse = RMSE(pred, true)
    mae = MAE(pred, true)
    eval_metrics.loc["%s" % rs]["test_R2"] = r2
    eval_metrics.loc["%s" % rs]["test_RMSE"] = rmse
    eval_metrics.loc["%s" % rs]["test_MAE"] = mae
    print("rs=%s\tR2=%.3f\tRMSE=%.4f\tMAE=%.4f\n" % (rs, r2, rmse, mae))
    f.write("\nSummary:\n")
    f.write("R2=%.3f\nRMSE=%.4f\nMAE=%.4f\n" % (r2, rmse, mae))
    f.close()

# Evaluation metrics report
for i in range(len(rs_list), eval_metrics.shape[0], len(rs_list)+1):
  for j in range(eval_metrics.shape[1]):
    eval_metrics.iloc[i,j] = "%.4f ± %.4f" % (eval_metrics.iloc[i-len(rs_list):i-1,j].mean(), eval_metrics.iloc[i-len(rs_list):i-1,j].std())
eval_metrics.to_csv("%s/exp/Heck_Other/SSTS_report_Chem(I=%s,R=%s)_%s.csv" % (doc_name, inter_intra, alpha_beta, datetime.datetime.now()))
print(eval_metrics)